# 6.3 Guardrails: validation, PII, cost/latency budgets

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Your API key - pick ONE option

Some cells in this notebook call a model, and you need your own API key. Run only ONE of the next two cells, whichever fits you:

**Option A - Gemini (free, recommended if you have no OpenAI credit):** Google AI Studio gives a free API key, no card needed. Open https://aistudio.google.com/apikey , create a key, then run the Gemini cell below. The Flash models run on a free tier with rate limits.

**Option B - OpenAI (paid):** create a key at https://platform.openai.com/api-keys and run the OpenAI cell below.

Whichever key you paste is stored only in this notebook session, never saved to the file.

In [ ]:
# Option A: Gemini (free). Run this cell OR the OpenAI one, not both.
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini (AI Studio) API key (input stays hidden): ")

In [ ]:
# Option B: OpenAI (paid). Run this cell OR the Gemini one, not both.
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

**Prerequisites:** 2.2 (custom tools), 2.4 (middleware: `ModelCallLimitMiddleware`, `ToolCallLimitMiddleware`, `recursion_limit` as the hard backstop). Every demo in this lesson runs with no API key, on a stub model.

## Learning objectives

- Layer guardrails as defence in depth: validate inputs before they reach the model, validate outputs before they reach the world.
- Reject malformed tool arguments with Pydantic schemas, and enforce structured output strictly with `ConfigDict(extra="forbid")`.
- Detect and redact PII with `PIIMiddleware` (and know exactly what its built-ins do *not* cover), including a custom regex detector and output-side redaction.
- Cap cost with `ModelCallLimitMiddleware` and `ToolCallLimitMiddleware`, and understand why call caps are not token budgets.
- Enforce latency with per-tool timeouts inside the tools themselves, plus an overall run deadline, and decide fail-closed versus degrade-gracefully when a budget trips.

## Defence in depth

A guardrail is a control that stands between an untrusted input and a consequential action. Agents need several, because no single layer covers every failure. The model can be jailbroken, so input filtering alone is not enough. The model can hallucinate a tool call, so output checking is needed too. The tool can hang, so timeouts are needed. The loop can spin, so budgets are needed. Each layer in this lesson is cheap, independent, and catches a different class of failure:

1. **Input validation:** reject malformed data before the model acts on it.
2. **Output validation:** check what the model produced before the world sees it.
3. **PII handling:** keep personal data out of model calls, tool calls, and logs.
4. **Cost budgets:** cap how much one run can spend.
5. **Latency budgets:** cap how long one run can take.

Skip any one layer and you inherit its failure mode. The airline chatbot that honoured a fabricated discount failed at layer 2. The support bot that pasted a customer's card number into a log failed at layer 3. The agent that burned a month of API budget over a weekend failed at layer 4. Practitioners teaching this topic converge on the same three-level picture: filter what goes in, check what comes out, and budget the whole run.

## Layer 1: input validation with Pydantic

Tool arguments are the most attacked surface of an agent: the model proposes them, and your code executes them. In LangChain 1.x, `@tool(args_schema=...)` validates the model's proposed arguments against a Pydantic model *before* the tool body runs. A bad call never reaches your code; it comes back to the model as a `ToolMessage` error it can learn from:

In [ ]:
# pip install langchain==1.4.6  (no API key needed)
from pydantic import BaseModel, ConfigDict, Field, ValidationError
from langchain_core.tools import tool

class RefundArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")   # reject unknown fields
    order_id: str = Field(pattern=r"^ORD-\d+$")
    amount_eur: float = Field(gt=0, le=5000)    # business rule, in the schema

@tool(args_schema=RefundArgs)
def issue_refund(order_id: str, amount_eur: float) -> str:
    """Issue a refund. Only reachable with valid arguments."""
    return f"refunded {amount_eur} EUR for {order_id}"

print(issue_refund.invoke({"order_id": "ORD-1042", "amount_eur": 49.99}))
# refunded 49.99 EUR for ORD-1042

try:
    issue_refund.invoke({"order_id": "nope", "amount_eur": -5, "hacker": True})
except ValidationError as e:
    print("rejected:", e.errors()[0]["type"], "|", e.errors()[1]["type"])
# rejected: string_pattern_mismatch | greater_than

Three details worth noticing. First, `extra="forbid"` rejects the `hacker` field: without it, Pydantic silently drops unknown fields, and a model stuffing extra arguments into your tool would never be told it is doing something odd. Second, the pattern and the range encode *business* rules, not just types: the schema is the cheapest place to say "refunds are at most 5,000 EUR". Third, the rejection happens before any side effect, which is the whole point: validation that runs after the refund is accounting, not a guardrail.

This also validates the model's output indirectly. When the model emits `{"order_id": "nope"}`, the tool node returns the validation error as a `ToolMessage`, and the model sees its mistake in context. Most models self-correct on the next turn. The schema is therefore both a guardrail and a feedback channel.

## Layer 2: output validation

Input validation protects your tools from the model. Output validation protects the user and the world from the model. Two mechanisms, used together.

**Strict structured output.** Lesson 1.3 taught `with_structured_output` with Pydantic. The guardrail version adds one line: `ConfigDict(extra="forbid")`. Verified on the pinned stack:

In [ ]:
class Answer(BaseModel):
    answer: str

class StrictAnswer(BaseModel):
    model_config = ConfigDict(extra="forbid")
    answer: str

Answer.model_validate({"answer": "x", "injected": "payload"})        # silently drops "injected"
StrictAnswer.model_validate({"answer": "x", "injected": "payload"})  # raises ValidationError

Without `forbid`, a model that returns extra fields has them silently discarded, which means you never learn that the model is improvising outside your contract. With it, the contract is enforced and violations are visible. Use the strict form for every schema that crosses a trust boundary: tool arguments, final answers shown to users, anything written to a database.

**Post-hoc checks on tool calls.** Schemas validate shape; they do not validate intent. A `fetch_url` tool with a perfect schema will still fetch `https://evil.example/x` if the model asks it to. The check belongs in a `wrap_tool_call` middleware, which sees every tool call before it executes:

In [ ]:
from urllib.parse import urlparse
from langchain_core.messages import ToolMessage
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call

ALLOWED_HOSTS = {"api.example.com", "docs.example.com"}

@wrap_tool_call
def url_allowlist(request, handler):
    url = request.tool_call.get("args", {}).get("url", "")
    host = urlparse(url).netloc
    if host and host not in ALLOWED_HOSTS:
        return ToolMessage(
            content=f"BLOCKED: host '{host}' is not on the allow-list.",
            tool_call_id=request.tool_call["id"],
            name=request.tool_call.get("name"),
        )
    return handler(request)

agent = create_agent(model, tools=[fetch_url], middleware=[url_allowlist])

Verified end to end on the pinned stack: the evil host never reaches the tool body, the model receives the BLOCKED message as the tool result, and allowed hosts pass through untouched. The same pattern enforces argument range checks the schema cannot express ("refunds above 500 EUR need the amount confirmed in the conversation first") and tool allow-lists per deployment ("this deployment may not call `send_email` at all").

## Layer 3: PII handling

Personal data flows through agents in three directions: user input going in, model output going out, and tool results in the middle. `PIIMiddleware`, at `langchain.agents.middleware`, handles all three with one declaration per PII type. Verified on the pinned stack:

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model,
    tools=[...],
    middleware=[
        PIIMiddleware("email", strategy="redact"),                 # input side
        PIIMiddleware("credit_card", strategy="mask"),             # keep last 4 readable
        PIIMiddleware("email", strategy="redact", apply_to_output=True),  # output side
    ],
)

Strategies: `block` raises `PIIDetectionError` (fail closed: the run stops); `redact` replaces with `[REDACTED_EMAIL]`; `mask` keeps partial readability (`************1111`); `hash` replaces with a deterministic pseudonym (`<email_hash:a1b2c3d4>`) so analytics still join on the same value without storing the address. Custom types use a regex detector:

In [ ]:
PIIMiddleware("api_key", detector=r"sk-[a-zA-Z0-9]{20,}", strategy="redact")
# model receives: "use key [REDACTED_API_KEY] for the call"

Verified behaviours you should know precisely:

- Built-in types are `email`, `credit_card` (Luhn-validated), `ip`, `mac_address`, and `url`. That is the whole list. **Names and phone numbers are not covered**: a phone number passes through untouched. If your threat model includes names, use a custom regex for your ID formats, or step up to an NER-based detector like Microsoft Presidio (spaCy NER plus regex plus checksums, 50+ entity types), which is the production standard practitioners reach for.
- `apply_to_input=True` is the default; set `apply_to_output=True` to scrub model output before the user sees it, and `apply_to_tool_results=True` for tool outputs (tool results are a classic leak path: the CRM returns the full customer record, the model quotes it).
- Redaction happens *before* the provider call, which is the property that matters: the third-party API never receives the raw PII at all.

Two policy rules sit above the code. First, **never send raw PII to a third-party API when the task does not need it**: strip or pseudonymise first, and re-identify inside your own perimeter if the final step needs the real value. Second, **write down a log redaction policy**: decide which fields may appear in traces and logs, apply the same middleware to the logging path, and keep the raw-value mapping (if you keep one) in a separate store with its own retention. Lesson 6.1's tracing makes every run inspectable, which is wonderful for debugging and terrible if the traces contain card numbers.

## Layer 4: cost budgets

An agent loop with no budget is a while-true with a credit card attached. The pinned stack gives you three controls, and one honest gap.

**`ModelCallLimitMiddleware`** caps model calls per run or per thread (verified end to end):

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware

middleware = [
    ModelCallLimitMiddleware(run_limit=20, exit_behavior="end"),   # or "error"
    ModelCallLimitMiddleware(thread_limit=200, exit_behavior="error"),
]

With `exit_behavior="end"` (the default), a tripped limit ends the run gracefully: the final message reads `Model call limits exceeded: run limit (20/20)`, which the user actually sees. With `"error"`, it raises `ModelCallLimitExceededError` for your code to catch. End is the default because a runaway agent answering "I hit my call budget" is better than a crashed request, but choose deliberately: for a billing-critical path, `error` plus your own handling is the honest choice.

**`ToolCallLimitMiddleware`** caps executions of one tool (or all tools, if `tool_name` is omitted):

In [ ]:
from langchain.agents.middleware import ToolCallLimitMiddleware

ToolCallLimitMiddleware(tool_name="web_search", run_limit=5, exit_behavior="end")
# exit_behavior: "continue" (skip the call, default) | "end" | "error"

Verified: with `run_limit=1`, the first call executes and the second attempt trips the limit with `'web_search' tool call limit reached: run limit exceeded (2/1 calls)`. Note the semantics: the limit counts completed calls, and the check fires *before* the call that would exceed it.

**The honest gap: call caps are not token budgets.** Nothing in the pinned stack bounds *token* consumption. Ten model calls with a 200k-token context cost far more than fifty calls with a short one, and `ModelCallLimitMiddleware` cannot see the difference. A `TokenBudgetMiddleware` reading `AIMessage.usage_metadata` after each model call has been proposed upstream but had not shipped as of October 2026, so teams hand-roll it as an `after_model` hook that accumulates input/output tokens and jumps to end when the budget is spent. Teach the gap explicitly: set call limits *and* watch token usage in LangSmith until a first-party budget middleware lands.

**`recursion_limit` is the hard backstop** (lesson 2.4): pass it in the invoke config, default 25, exceeding it raises `GraphRecursionError` from `langgraph.errors`. Roughly two super-steps per ReAct iteration, so 25 is about 12 tool round-trips. It is a backstop, not a budget: hitting it means something already went wrong.

## Layer 5: latency budgets

There is no built-in per-tool timeout in the middleware suite (verified: the option does not exist on the pinned stack). Timeouts live inside the tools, where the I/O actually happens:

In [ ]:
import asyncio
from langchain_core.tools import tool

@tool
async def lookup_order(order_id: str) -> str:
    """Look up an order; never hangs longer than 8 seconds."""
    try:
        return await asyncio.wait_for(call_warehouse_api(order_id), timeout=8.0)
    except asyncio.TimeoutError:
        return "TIMEOUT: the warehouse API did not respond; try again or narrow the query"

Verified with `agent.ainvoke`: the tool returns the timeout message as a normal `ToolMessage`, the agent reads it, and the run continues gracefully instead of hanging. Two notes: async tools need `ainvoke` (sync `invoke` on an async-only tool raises, it does not magically run the coroutine), and for sync HTTP code the equivalent is `httpx.Timeout(8.0)` on the client. Return the timeout *as data* rather than raising where you can: a raised exception aborts the tool call, while a timeout message lets the model retry, rephrase, or tell the user what happened.

For the whole run, wrap the invocation in a deadline:

In [ ]:
try:
    result = await asyncio.wait_for(agent.ainvoke({"messages": messages}), timeout=120.0)
except asyncio.TimeoutError:
    result = {"messages": [{"role": "assistant",
        "content": "This is taking longer than two minutes; I have stopped to avoid a runaway run."}]}

## When a budget trips: fail closed or degrade gracefully

Every budget needs a pre-decided answer to one question: when it trips, does the run stop loudly or finish quietly? The middleware gives you both verbs; you supply the policy:

| Situation | Recommended | Why |
|---|---|---|
| Model-call budget hit mid-task | `end` with the budget message as the final answer | the user sees what happened; no silent truncation |
| PII `block` strategy triggered | `error` (`PIIDetectionError`) | fail closed: personal data must not flow on a maybe |
| Disallowed tool host | block the call, return the reason as the tool result | the model can try a legitimate alternative |
| Tool timeout | return the timeout as data | the model retries or explains; the run survives |
| Overall run deadline | stop and say so explicitly | a hung run that "finishes" with a partial answer is a lie |

Whichever you choose, **tell the user which happened**. The verified `end` behaviour does this for free: the limit message becomes the final assistant message. The failure mode to avoid is the silent one: a run that stops at 20 calls and presents the half-finished draft as the answer. Budgets that do not announce themselves become misinformation.

## The capstone guardrail checklist

Copy this into your capstone README and check each item before you deploy (lesson 6.4):

- [ ] Tool schemas reject malformed arguments (`args_schema` with ranges/patterns; `extra="forbid"`).
- [ ] Structured outputs use `ConfigDict(extra="forbid")` on every trust boundary.
- [ ] `wrap_tool_call` allow-lists for URLs/hosts; per-deployment tool allow-lists.
- [ ] `PIIMiddleware` on input for every PII type you handle; `apply_to_output=True` where outputs reach users; `apply_to_tool_results=True` where tools return customer records.
- [ ] Custom detectors for your ID formats (order numbers, national IDs); Presidio evaluated if names/phones matter.
- [ ] Log redaction policy written down; raw PII never in traces sent to third parties.
- [ ] `ModelCallLimitMiddleware(run_limit=...)` set from a cost model, not a guess.
- [ ] `ToolCallLimitMiddleware` on every paid or slow tool.
- [ ] Per-tool timeouts inside every network tool; overall run deadline around `ainvoke`.
- [ ] Trip policy decided per budget: `end` versus `error`, and the user is told.

## EU angle: minimisation and documented controls

Two GDPR and AI Act threads run through this lesson. First, **data minimisation** (GDPR Article 5): an agent should hold the least personal data that lets it do its job. `PIIMiddleware` with redact-before-the-provider-call is minimisation implemented, not just promised; the log redaction policy is the evidence that you thought about the traces too. Lesson 6.5 covers retention and residency for those traces. Second, **operational controls a deployer can document**: the AI Act expects deployers to monitor their systems. Cost and latency budgets with trip policies are exactly the kind of operational control an auditor understands: "no run may exceed 20 model calls; when it does, the run ends and the user is told." Write the numbers down, keep the experiment history from lesson 6.2, and the compliance story largely writes itself.

## What can go wrong

- **Schema without `extra="forbid"`.** Unknown fields are silently dropped, so a model improvising outside your contract never gets corrected and you never notice. Forbid on every trust boundary.
- **Validating shape but not intent.** The schema says the URL is a string; it does not say the host is allowed. Shape checks and allow-lists are different layers; you need both.
- **PIIMiddleware's blind spots.** Names and phone numbers pass through the built-ins untouched. If your threat model includes them and you only configured the middleware, you have a guardrail-shaped hole. Add custom detectors or Presidio.
- **Redacting after the provider call.** A middleware ordered after the model call, or a log pipeline that stores the raw input, defeats the purpose. Redact before the data leaves your perimeter, in every direction.
- **Call caps mistaken for cost caps.** Ten calls with a huge context can cost more than fifty short ones. Until a first-party token budget middleware ships, pair call limits with token monitoring in LangSmith.
- **No per-tool timeout.** The middleware suite has none; a hanging tool hangs the run. Timeouts live inside the tools (`asyncio.wait_for`, `httpx.Timeout`), not in the agent config.
- **Silent budget trips.** A run that ends at the call limit and presents a half-finished answer as complete is misinformation. Always surface the trip: the verified `end` message does this, but only if you do not swallow it.
- **`error` without a handler.** `exit_behavior="error"` raises into your code. If nothing catches `ModelCallLimitExceededError`, the user gets a 500 instead of an explanation. Catch it and translate it.
- **Timeouts that raise instead of returning.** A raised `TimeoutError` aborts the tool call; a timeout returned as data lets the model retry or explain. Prefer returning, unless the timeout means the run must stop.

## Check your understanding

**1. A tool has `args_schema` with `Field(gt=0, le=5000)` and `extra="forbid"`. The model calls it with `{"amount_eur": -5, "extra": 1}`. What happens, and why does the rejection occur before any side effect?**
The tool node validates the arguments against the schema before the tool body runs, so the `ValidationError` fires first and the refund is never issued. The error returns to the model as a `ToolMessage`, giving it a chance to self-correct. Validation-before-execution is what makes the schema a guardrail rather than accounting.

**2. Why is `ConfigDict(extra="forbid")` a guardrail, not just tidiness? Give one input-side and one output-side example.**
Without it, extra fields are silently dropped, so contract violations are invisible. Input side: a model stuffing an unexpected `hacker` argument into `issue_refund` would never be told it is improvising. Output side: a final-answer schema that silently drops an `injected` field hides the fact that the model is returning data outside your contract. `forbid` turns both into loud errors.

**3. `PIIMiddleware("email", strategy="redact")` is configured. A user writes "call me on +92 300 1234567". What reaches the model, and what does this reveal about the middleware's coverage?**
The phone number reaches the model unredacted, because the built-in detectors cover only email, credit card, IP, MAC address, and URL. Names and phone numbers are not covered. The lesson: know the exact detector list, and add custom regex detectors or an NER-based tool like Presidio for the types your threat model includes.

**4. Your agent averages 8 model calls per run but one run makes 200 calls against a 200k-token context. Which budget catches it, and what is the gap in the pinned stack?**
`ModelCallLimitMiddleware(run_limit=...)` catches the 200-call run. The gap: nothing in the pinned stack bounds token consumption, so a few calls with a huge context can cost more than many short calls without tripping any limit. Mitigate with call limits plus token monitoring in LangSmith, or a hand-rolled `after_model` hook accumulating `usage_metadata`, until a first-party token budget middleware ships.

**5. A warehouse API tool hangs for 90 seconds. Where does the timeout belong, what should the tool return on timeout, and why not raise?**
The timeout belongs inside the tool (`asyncio.wait_for` for async code, `httpx.Timeout` for sync HTTP), because the middleware suite has no per-tool timeout. On timeout the tool should return the timeout as data (a normal tool result saying the API did not respond), because a returned message lets the model retry, rephrase, or explain to the user, while a raised exception aborts the tool call and usually the run's graceful recovery.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.